# 01 · Funnel EDA — where do conversions come from?

**Question for the sales manager:** which leads convert, and which are a waste of a call?
Every chart title below is a finding computed from the data (built from the tables, not typed by hand).
Logic lives in `src/`; this notebook only calls it.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display
from src.config import FIG_DIR, ROOT as REPO
from src.clean import load_raw, fix_case_duplicates, LEAKY_COLS
from src import eda

pd.set_option("display.float_format", "{:.3f}".format)
FIG_DIR.mkdir(parents=True, exist_ok=True)
FINDINGS = {}

## 1. Baseline

In [2]:
df = fix_case_duplicates(load_raw())   # "Select" -> NaN; google -> Google
BASE = eda.baseline_rate(df)
print(f"{len(df):,} leads, {df.Converted.sum():,} converted -> baseline conversion rate {BASE:.1%}")
print(f"Do Not Call = Yes: {(df['Do Not Call']=='Yes').sum()} leads")

9,240 leads, 3,561 converted -> baseline conversion rate 38.5%
Do Not Call = Yes: 2 leads


## 2. Lead Origin

In [3]:
t = eda.segment_table(df, "Lead Origin")
display(t)
title = eda.headline_best_vs_biggest(t, "leads")
eda.plot_segments(t, BASE, title, FIG_DIR / "01_origin.png")
best = t.loc["Lead Add Form"]
FINDINGS["origin"] = (f"Lead Add Form leads convert at {best.rate:.0%} ({best.lift:.1f}× baseline) "
                      f"but are only {best.share:.0%} of volume.")

,leads,conversions,rate,share,lift
Lead Origin,,,,,
Landing Page Submission,4886,1768,0.362,0.529,0.939
API,3580,1115,0.311,0.387,0.808
Lead Add Form,718,664,0.925,0.078,2.400
Other,56,14,0.250,0.006,0.649


## 3. Lead Source — rate, volume and the quadrant

In [4]:
t_src = eda.segment_table(df, "Lead Source")
display(t_src)
eda.plot_segments(t_src, BASE, eda.headline_best_vs_biggest(t_src, "leads"),
                  FIG_DIR / "01_source.png")

q = eda.quadrant_table(t_src)
display(q[["leads", "share", "rate", "lift", "quadrant"]])
fix = q[q.quadrant == "fix or filter"].sort_values("leads", ascending=False)
grow = q[q.quadrant == "grow"].sort_values("lift", ascending=False)
fix_lifts = " and ".join(f"{x:.2f}×" for x in fix.lift)
qtitle = (f"{' and '.join(fix.index)} bring {fix.share.sum():.0%} of leads but convert at only "
          f"{fix_lifts} baseline; {' and '.join(grow.index[:2])} convert at "
          f"{grow.lift.iloc[:2].min():.1f}–{grow.lift.iloc[:2].max():.1f}× but are "
          f"{grow.share.iloc[:2].sum():.0%} of volume combined")
eda.plot_quadrant(q, BASE, qtitle, FIG_DIR / "01_source_quadrant.png")
FINDINGS["fix_sources"] = (
    f"{' and '.join(fix.index)} bring {fix.share.sum():.0%} of leads but convert at "
    + " and ".join(f"{r:.0%}" for r in fix.rate) + f" vs a {BASE:.0%} baseline.")

,leads,conversions,rate,share,lift
Lead Source,,,,,
Google,2873,1147,0.399,0.311,1.036
Direct Traffic,2543,818,0.322,0.275,0.835
Olark Chat,1755,448,0.255,0.190,0.662
Organic Search,1154,436,0.378,0.125,0.980
Reference,534,490,0.918,0.058,2.381
Welingak Website,142,140,0.986,0.015,2.558
Referral Sites,125,31,0.248,0.014,0.644
Other,78,22,0.282,0.008,0.732
Missing,36,29,0.806,0.004,2.090


,leads,share,rate,lift,quadrant
Lead Source,,,,,
Google,2873,0.311,0.399,1.036,scale
Direct Traffic,2543,0.275,0.322,0.835,fix or filter
Olark Chat,1755,0.190,0.255,0.662,fix or filter
Organic Search,1154,0.125,0.378,0.980,scale
Reference,534,0.058,0.918,2.381,grow
Welingak Website,142,0.015,0.986,2.558,grow
Referral Sites,125,0.014,0.248,0.644,deprioritise


In [5]:
# Reference and Welingak Website are sub-channels of Lead Add Form (check):
pd.crosstab(df["Lead Source"], df["Lead Origin"]).loc[["Reference", "Welingak Website"]]

Lead Origin,API,Landing Page Submission,Lead Add Form,Lead Import
Lead Source,,,,
Reference,0,0,534,0
Welingak Website,0,0,142,0


## 4. Occupation

In [6]:
occ = "What is your current occupation"
t = eda.segment_table(df, occ)
display(t)
wp, ms = t.loc["Working Professional"], t.loc[eda.MISSING_LABEL]
title = (f"Working professionals convert at {wp.rate:.0%} ({wp.lift:.1f}× baseline) but are {wp.share:.0%} "
         f"of leads; the {ms.share:.0%} who leave occupation blank convert at just {ms.rate:.0%}")
eda.plot_segments(t, BASE, title, FIG_DIR / "01_occupation.png")
FINDINGS["occupation"] = title + "."

,leads,conversions,rate,share,lift
What is your current occupation,,,,,
Unemployed,5600,2441,0.436,0.606,1.131
Missing,2690,370,0.138,0.291,0.357
Working Professional,706,647,0.916,0.076,2.378
Student,210,78,0.371,0.023,0.964
Other,34,25,0.735,0.004,1.908


## 5. Specialization

In [7]:
t = eda.segment_table(df, "Specialization")
display(t)
named = t[~t.index.isin([eda.OTHER, eda.MISSING_LABEL])]
ms = t.loc[eda.MISSING_LABEL]
present_rate = df.loc[df.Specialization.notna(), "Converted"].mean()
title = (f"Naming any specialization matters more than which one: named {present_rate:.0%} vs "
         f"{ms.rate:.0%} for the {ms.share:.0%} left blank; no single specialization exceeds "
         f"{named.lift.max():.1f}× baseline")
eda.plot_segments(t, BASE, title, FIG_DIR / "01_specialization.png", top=18)

,leads,conversions,rate,share,lift
Specialization,,,,,
Missing,3380,969,0.287,0.366,0.744
Finance Management,976,436,0.447,0.106,1.159
Human Resource Management,848,388,0.458,0.092,1.187
Marketing Management,838,408,0.487,0.091,1.263
Operations Management,503,238,0.473,0.054,1.228
Business Administration,403,179,0.444,0.044,1.153
IT Projects Management,366,140,0.383,0.040,0.993
Supply Chain Management,349,151,0.433,0.038,1.123
"Banking, Investment And Insurance",338,167,0.494,0.037,1.282


<Figure size 800x854 with 1 Axes>

## 6. City

In [8]:
t = eda.segment_table(df, "City")
display(t)
named = t[~t.index.isin([eda.OTHER, eda.MISSING_LABEL])]
title = (f"City barely moves conversion: every named city group is within "
         f"{named.lift.min():.2f}–{named.lift.max():.2f}× baseline")
eda.plot_segments(t, BASE, title, FIG_DIR / "01_city.png")

,leads,conversions,rate,share,lift
City,,,,,
Missing,3669,1257,0.343,0.397,0.889
Mumbai,3222,1309,0.406,0.349,1.054
Thane & Outskirts,752,338,0.449,0.081,1.166
Other Cities,686,276,0.402,0.074,1.044
Other Cities of Maharashtra,457,201,0.440,0.049,1.141
Other Metro Cities,380,155,0.408,0.041,1.058
Other,74,25,0.338,0.008,0.877


<Figure size 800x434 with 1 Axes>

## 7. Engagement: time on site and visits

In [9]:
tt = eda.binned_table(df, "Total Time Spent on Website",
                      bins=[0, 1, 60, 300, 600, 1200, float("inf")],
                      labels=["0s", "<1 min", "1–5 min", "5–10 min", "10–20 min", "20+ min"])
display(tt)
lo, hi = tt.loc["1–5 min"], tt.loc[["10–20 min", "20+ min"]]
hi_rate = hi.conversions.sum() / hi.leads.sum()
title = (f"Leads who spend 10+ minutes on the site convert at {hi_rate:.0%} vs {lo.rate:.0%} at 1–5 min "
         f"({hi.share.sum():.0%} of leads are in the 10+ min group)")
FINDINGS["time"] = title + "."

# Why does the 0s bar look good? Mix effect: it includes Lead Add Form leads,
# who never browse the site but convert very well.
zero = df["Total Time Spent on Website"] == 0
mix = eda.mix_within(df, zero, "Lead Origin")
display(mix)
note = (f"The 0s bar is a mix effect, not a signal: {mix.loc['Lead Add Form','share_of_subset']:.0%} of it is "
        f"Lead Add Form leads (convert at {mix.loc['Lead Add Form','rate']:.0%}); API leads with 0s on site "
        f"convert at only {mix.loc['API','rate']:.0%}.")
print(note)
eda.plot_binned(tt, BASE, title, "Total time spent on website", FIG_DIR / "01_time_on_site.png", note=note)

,leads,conversions,rate,share,lift
Total Time Spent on Website,,,,,
0s,2193,916,0.418,0.237,1.084
<1 min,602,84,0.140,0.065,0.362
1–5 min,2242,312,0.139,0.243,0.361
5–10 min,1250,237,0.190,0.135,0.492
10–20 min,1456,976,0.670,0.158,1.739
20+ min,1497,1036,0.692,0.162,1.796


,leads,rate,share_of_subset
Lead Origin,,,
API,1603,0.242,0.731
Lead Add Form,557,0.932,0.254
Lead Import,30,0.300,0.014
Landing Page Submission,3,0.000,0.001


The 0s bar is a mix effect, not a signal: 25% of it is Lead Add Form leads (convert at 93%); API leads with 0s on site convert at only 24%.


<Figure size 800x450 with 1 Axes>

In [10]:
tv = eda.binned_table(df, "TotalVisits", bins=[0, 1, 2, 4, 6, 11, float("inf")],
                      labels=["0", "1", "2–3", "4–5", "6–10", "11+"])
display(tv)
multi = tv.loc[["2–3", "4–5", "6–10", "11+"]]
title = (f"Visit count matters less than time spent: 2+ visits convert at "
         f"{multi.lift.min():.2f}–{multi.lift.max():.2f}× baseline; a single visit at {tv.loc['1'].lift:.2f}×")
eda.plot_binned(tv, BASE, title, "Total visits", FIG_DIR / "01_visits.png")

,leads,conversions,rate,share,lift
TotalVisits,,,,,
0,2189,916,0.418,0.237,1.086
1,395,60,0.152,0.043,0.394
2–3,2986,1023,0.343,0.323,0.889
4–5,1903,764,0.401,0.206,1.042
6–10,1277,533,0.417,0.138,1.083
11+,353,165,0.467,0.038,1.213
Missing,137,100,0.730,0.015,1.894


<Figure size 800x450 with 1 Axes>

## 8. Missingness — is a blank field itself a signal?

In [11]:
miss = eda.missingness_table(df)
miss_nz = miss[miss.pct_missing > 0]
display(miss_nz)
miss_nz.to_csv(REPO / "reports" / "missingness.csv")

# Headline only from fields known at lead creation (exclude post-contact columns).
usable = miss_nz[(~miss_nz.index.isin(LEAKY_COLS)) & (miss_nz.p_value < 0.01)].sort_values("gap_pp")
a, b = usable.index[:2]
title = (f"A blank form field is itself a signal: leads who skip '{a}' or '{b}' convert "
         f"~{abs(usable.gap_pp.iloc[:2]).mean():.0f} pp below those who answer")
eda.plot_missingness(miss_nz, title, FIG_DIR / "01_missingness.png", top=17, leaky=LEAKY_COLS)
both = df[a].isna() & df[b].isna()
print(f"'{a}' and '{b}' are blank together for {both.sum():,} leads "
      f"({both.sum() / df[a].isna().sum():.0%} of leads missing '{a}').")

,pct_missing,rate_if_missing,rate_if_present,gap_pp,p_value
column,,,,,
How did you hear about X Education,0.785,0.374,0.426,-5.128,0.000
Lead Profile,0.742,0.301,0.629,-32.771,0.000
Lead Quality,0.516,0.215,0.567,-35.237,0.000
Asymmetrique Profile Index,0.456,0.392,0.380,1.196,0.248
Asymmetrique Activity Index,0.456,0.392,0.380,1.196,0.248
Asymmetrique Profile Score,0.456,0.392,0.380,1.196,0.248
Asymmetrique Activity Score,0.456,0.392,0.380,1.196,0.248
City,0.397,0.343,0.414,-7.097,0.000
Specialization,0.366,0.287,0.442,-15.563,0.000


'What matters most to you in choosing a course' and 'What is your current occupation' are blank together for 2,690 leads (99% of leads missing 'What matters most to you in choosing a course').


Note: `Lead Quality`, `Lead Profile`, `Tags`, `Last Activity` and the `Asymmetrique` columns appear above but
are post-contact / sales-assigned and will be dropped from the model in Phase 2 (leakage).

## 9. Do Not Email

In [12]:
t = eda.segment_table(df, "Do Not Email")
display(t)
y = t.loc["Yes"]
FINDINGS["dne"] = (f"Leads who opt out of email ({y.share:.0%} of volume) convert at {y.rate:.0%} vs "
                   f"{t.loc['No'].rate:.0%} for the rest.")

,leads,conversions,rate,share,lift
Do Not Email,,,,,
No,8506,3443,0.405,0.921,1.050
Yes,734,118,0.161,0.079,0.417


## 10. Top findings (computed)

In [13]:
print(f"Baseline conversion: {BASE:.1%} of {len(df):,} leads\n")
for i, k in enumerate(["origin", "occupation", "time", "fix_sources", "dne"], 1):
    print(f"{i}. {FINDINGS[k]}")

Baseline conversion: 38.5% of 9,240 leads

1. Lead Add Form leads convert at 92% (2.4× baseline) but are only 8% of volume.
2. Working professionals convert at 92% (2.4× baseline) but are 8% of leads; the 29% who leave occupation blank convert at just 14%.
3. Leads who spend 10+ minutes on the site convert at 68% vs 14% at 1–5 min (32% of leads are in the 10+ min group).
4. Direct Traffic and Olark Chat bring 47% of leads but convert at 32% and 26% vs a 39% baseline.
5. Leads who opt out of email (8% of volume) convert at 16% vs 40% for the rest.
